# Modified GANet vs. Original GANet Benchmark on Kaggle
### Continuous Sign Language Recognition on ISL-CSLTR

This notebook implements the complete **Modified GANet** pipeline on Kaggle with zero dataset download time:
1. **Transfer Only SMM (`transfer_gam: false`)**: Transfers the 97% spatial ResNet features from INCLUDE, but lets GCMM + GAM learn multi-sign sentence attention.
2. **VAC Knowledge Distillation (`loss.vac: 0.1`, `loss.kd: 0.5`)**: Combines mid-stage visual CTC with KL-divergence self-distillation.
3. **Paper's Exact LR Schedule (`milestones: [10, 20]`, `gamma: 0.2`, `batch_size: 2`)**: Steps down learning rate at Epochs 10 and 20 to prevent CTC blank relapse.
4. **Calibrated Temporal Masking (`mask_chunk: [2, 5]`, `mask_p: 0.25`, `mask_max_drop: 0.15`)**: Prevents whole-sign erasure.
5. **Zero-Leakage Signer-Independent Split**: Train on Signers 1-5 (474 clips), Dev on Signer 6 (96 clips), Test on Signer 7 (93 clips) with 100% vocabulary coverage.

### Step 1: Check GPU & Environment
Ensure GPU is enabled in the right panel under **Settings -> Accelerator -> GPU T4 x2 or GPU P100**.

In [ ]:
!nvidia-smi

### Step 2: Unpack Codebase
Upload `ganet_code.zip` to Kaggle (via **Add Data -> Upload Dataset** or git clone into `/kaggle/working/`).

In [ ]:
import os, glob

!mkdir -p /kaggle/working/ganet_isl

# Automatically locate and unpack ganet_code.zip from any uploaded Kaggle dataset
code_zips = glob.glob('/kaggle/input/**/ganet_code.zip', recursive=True)
if code_zips:
    print(f"Unpacking codebase from: {code_zips[0]}")
    !unzip -qo {code_zips[0]} -d /kaggle/working/ganet_isl/
else:
    print("Note: If ganet_code.zip is not uploaded as a dataset, make sure code is placed in /kaggle/working/ganet_isl/")

%cd /kaggle/working/ganet_isl
!pip install -q opencv-python pyyaml tqdm
print("[✓] Codebase ready at:", os.getcwd())

### Step 3: Verify Mounted Kaggle Datasets
Make sure the ISL-CSLTR dataset is attached to this notebook in Kaggle (right panel **Add Data** -> search for `drblack00/isl-csltr-indian-sign-language-dataset`):

In [ ]:
!ls -lh /kaggle/input/

### Step 4: Generate 663-Sample Manifest & Signer-Independent Split
Scans all 663 leaf frame directories under `Frames_Sentence_Level`, mapping sentences and real signers (`signer1` to `signer7`):
- **Train**: Signers 1–5 (474 clips)
- **Dev**: Signer 6 (96 clips)
- **Test**: Signer 7 (93 clips)
- **Zero vocabulary leakage**: 100% of all 175 words seen in train.

In [ ]:
%cd /kaggle/working/ganet_isl

# Automatically finds Frames_Sentence_Level anywhere in /kaggle/input/
!python tools/make_isl_cslrt_manifest.py \
    --search_root /kaggle/input \
    --manifest_out data/isl_manifest.csv \
    --splits_out data/isl_splits.json \
    --compat_splits splits/isl_signer_v1.json \
    --dev_signer signer6 \
    --test_signer signer7

# Verify CTC feasibility and vocabulary coverage
!python tools/verify_data.py --isl_manifest data/isl_manifest.csv --splits data/isl_splits.json --num_pool 2 --skip_include

### Step 5: Check Stage 1 Pre-trained Weights (`include_visual.pt`)
If you uploaded `include_visual.pt` or `include_visual.zip` as a Kaggle dataset, copy it into `work_dir/include_pretrain/`:

In [ ]:
%cd /kaggle/working/ganet_isl
import os, glob

!mkdir -p work_dir/include_pretrain

# Check if pre-trained weights were uploaded in an input dataset
pre_pts = glob.glob('/kaggle/input/**/include_visual.pt', recursive=True)
pre_zips = glob.glob('/kaggle/input/**/include_visual.zip', recursive=True)

if pre_zips:
    print(f"Unpacking pre-trained weights from: {pre_zips[0]}")
    !unzip -qo {pre_zips[0]} -d /kaggle/working/ganet_isl/
elif pre_pts:
    print(f"Copying pre-trained weights from: {pre_pts[0]}")
    !cp {pre_pts[0]} work_dir/include_pretrain/include_visual.pt

if not os.path.exists('work_dir/include_pretrain/include_visual.pt'):
    print("Pre-trained checkpoint not found in /kaggle/input/. Bootstrapping with ImageNet ResNet-18 weights...")
    from ganet.models.include_model import IncludeClassifier
    import torch
    model = IncludeClassifier(num_classes=8, gcmm=True, gam=True, imagenet=True, temporal_mix=True)
    torch.save(model.visual.state_dict(), 'work_dir/include_pretrain/include_visual.pt')

!ls -lh work_dir/include_pretrain/include_visual.pt
print("[✓] Stage 1 pre-trained weights verified!")

### Step 6: Run 1 — Original GANet Architecture Baseline (Trained on ISL-CSLTR)
Trains the exact original paper architecture (SMM + GCMM + GAM + Visual-tapped GLP, with full Stage 1 transfer) without Temporal Masking or Mid-Stage VAC/KD on the 663-sample ISL-CSLTR dataset:

In [ ]:
%cd /kaggle/working/ganet_isl

!python train_isl.py --config configs/isl_modified_ganet.yaml \
    --set work_dir=work_dir/isl_original_ganet \
          data.mask_p=0.0 \
          model.vac=false \
          model.glp_tap=visual \
          pretrained.transfer_gam=true \
          loss.vac=0.0 \
          loss.kd=0.0 \
          loss.glp=1.0

### Step 7: Run 2 — Proposed Modified GANet Architecture
Trains the full Modified GANet pipeline (SMM handshape transfer + newly initialized continuous GAM + calibrated Temporal Masking + Mid-Stage VAC/KD + BiLSTM-tapped GLP):

In [ ]:
%cd /kaggle/working/ganet_isl

!python train_isl.py --config configs/isl_modified_ganet.yaml \
    --set work_dir=work_dir/isl_modified_ganet_s3407

### Step 8: Compare Benchmark Results on Test Split (Signer 7)

In [ ]:
%cd /kaggle/working/ganet_isl
import json, os

print("=" * 65)
print("BENCHMARK COMPARISON ON TEST SPLIT (SIGNER 7)")
print("=" * 65)

for name, path in [
    ("Original GANet Baseline", "work_dir/isl_original_ganet/results.json"),
    ("Proposed Modified GANet", "work_dir/isl_modified_ganet_s3407/results.json")
]:
    if os.path.exists(path):
        data = json.load(open(path))
        t = data.get('test', {})
        print(f"{name:28s} | Test WER: {t.get('wer', 0.0):.2f}% | Del: {t.get('del_rate', 0.0):.2f}% | Sub: {t.get('sub_rate', 0.0):.2f}%")
    else:
        print(f"{name:28s} | Results not found at {path}")